# Bayesian DIF 검출 튜토리얼
## 이분형 응답 데이터, 1PL/2PL IRT, Stan/cmdstanpy

본 노트북은 *dif_topics.md* 보고서의 §3.3(Bayesian DIF) 및 §5(이분형 응답 데이터 기반 Bayesian DIF) 절에 대응하는 실습 튜토리얼이다.

### 학습 목표
1. **IRT 기본 모형(1PL/Rasch, 2PL)** 의 수식과 ICC(Item Characteristic Curve) 를 시각적으로 복습한다.
2. **균일 DIF(uniform DIF)** 와 **비균일 DIF(non-uniform DIF)** 의 정의·도식을 비교한다.
3. 알려진 DIF 문항을 포함한 **이분형 응답 자료를 시뮬레이션**한다.
4. **Zumbo (1999)** 식 **로지스틱 회귀 기반 DIF 절차**로 의심 문항을 1차 스크리닝한다.
5. **Stan / cmdstanpy** 로 **Bayesian 1PL·2PL DIF 모형**을 추정하여 사후확률분포로부터 DIF 문항을 검출한다.
6. **Joo & Lee (2022)** 가 제안한 **PPMC 기반 DIF 검출** 절차를 구현하여 시연한다.

### 사용 패키지
`numpy, pandas, matplotlib, seaborn, scipy, statsmodels, cmdstanpy` (Stan toolchain 설치 필요).

> 모든 산출물(Stan 파일·노트북)은 파일명이 **`tut_`** 로 시작한다.


---
## 0. 환경 준비

먼저 분석에 필요한 패키지를 불러오고, 재현성을 위한 난수 시드를 설정한다. Stan 모형 파일들은 `tut_stan_models/` 폴더에 위치한다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

RNG_SEED = 2026
rng = np.random.default_rng(RNG_SEED)

STAN_DIR = "tut_stan_models"
assert os.path.isdir(STAN_DIR), f"{STAN_DIR} 폴더가 보이지 않는다. 노트북과 같은 위치에 두어야 한다."
print("환경 준비 완료. Stan 모형 폴더:", os.listdir(STAN_DIR))

---
## 1. IRT 기본 모형 복습 (1PL · 2PL)

### 1.1 Rasch / 1PL 모형
1모수 로지스틱 모형은 능력 $\theta$ 와 문항 난이도 $b_j$ 만으로 정답확률을 모델링한다.

$$P(Y_{ij}=1\mid \theta_i, b_j)=\frac{\exp(\theta_i-b_j)}{1+\exp(\theta_i-b_j)}=\sigma(\theta_i-b_j)$$

모든 문항의 **변별도(기울기)** 가 동일하므로 ICC 들이 서로 **평행** 하다.

### 1.2 2PL 모형
2모수 로지스틱 모형은 문항별로 변별도 $a_j>0$ 가 다르다.

$$P(Y_{ij}=1\mid \theta_i, a_j, b_j)=\sigma\!\big(a_j(\theta_i-b_j)\big)$$

- $b_j$ : 정답확률이 0.5 가 되는 능력 수준(난이도)
- $a_j$ : ICC 의 변곡점 기울기(변별도)

문항마다 변별도가 다르면 ICC 들이 서로 다른 기울기를 가지며, 일부 곡선은 **교차** 할 수도 있다.


아래 코드는 동일한 능력 격자 $\theta\in[-4,4]$ 위에서 1PL 과 2PL 의 ICC 를 그려 두 모형의 외형 차이를 시각화한다. `expit` 은 시그모이드 함수이다.

In [ ]:
theta_grid = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

# --- 1PL: 동일한 변별도, 난이도만 변경 ---
ax = axes[0]
for b in [-1.5, -0.5, 0.5, 1.5]:
    ax.plot(theta_grid, expit(theta_grid - b), label=f"b={b:+.1f}")
ax.set_title("1PL (Rasch) — 변별도 동일, 곡선 평행")
ax.set_xlabel(r"능력 $\theta$"); ax.set_ylabel(r"$P(Y=1)$")
ax.axhline(0.5, color="gray", lw=0.7, ls="--"); ax.legend(fontsize=8)

# --- 2PL: 문항별 변별도와 난이도 모두 변경 ---
ax = axes[1]
for a, b, c in [(0.6, -1.0, "tab:blue"), (1.0, 0.0, "tab:orange"),
                (1.7, 0.8, "tab:green"), (2.5, -0.3, "tab:red")]:
    ax.plot(theta_grid, expit(a * (theta_grid - b)),
            color=c, label=f"a={a}, b={b:+.1f}")
ax.set_title("2PL — 변별도·난이도 모두 다름")
ax.set_xlabel(r"능력 $\theta$")
ax.axhline(0.5, color="gray", lw=0.7, ls="--"); ax.legend(fontsize=8)

plt.suptitle("그림 1.  1PL 과 2PL 의 문항특성곡선(ICC) 비교", y=1.03)
plt.tight_layout(); plt.show()

---
## 2. Uniform DIF 와 Non-uniform DIF

DIF(Differential Item Functioning, 차별기능문항) 은 **동일한 잠재 능력을 가진 두 집단의 정답확률이 문항 수준에서 체계적으로 달라지는 현상** 이다. 표준적 용어로 능력 분포가 기준이 되는 집단을 **준거집단(reference)**, 비교 대상을 **초점집단(focal)** 이라 한다.

### 2.1 균일 DIF (Uniform DIF)
모든 능력 수준에서 한 집단이 **일관되게** 유리(불리)한 경우이다. 1PL 식으로 표현하면 문항 난이도가 집단별로 다음과 같이 다르다:

$$\operatorname{logit} P(Y_{ij}=1)=\theta_i-(b_j+\delta^{b}_j\, g_i),\qquad g_i\in\{0,1\}$$

- $g_i=0$ : 준거집단,  $g_i=1$ : 초점집단
- $\delta^{b}_j$ : 초점집단에 대한 **난이도 이동(uniform DIF)** — 양수이면 초점집단이 더 어렵게 답한다는 의미
- 두 집단의 ICC 는 서로 **평행 이동(horizontal shift)** 한다.

### 2.2 비균일 DIF (Non-uniform DIF)
능력 수준에 따라 어느 집단이 유리한지가 **달라지는** 경우이다. 2PL 로 확장하면 변별도까지 집단별로 다르다:

$$\operatorname{logit} P(Y_{ij}=1)=(a_j+\delta^{a}_j\, g_i)\,\bigl(\theta_i-(b_j+\delta^{b}_j\, g_i)\bigr)$$

- $\delta^{a}_j\neq 0$ 이면 두 집단의 ICC 기울기가 다르므로 **교차(crossing)** 가 발생한다.
- 균일+비균일 DIF 가 동시에 있는 경우도 흔하다.

다음 코드 셀은 두 가지 DIF 패턴을 한 그림에 시각화한다.

In [ ]:
theta_grid = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

# --- (a) Uniform DIF: 평행 이동 ---
b, delta_b = 0.0, 0.8
ax = axes[0]
ax.plot(theta_grid, expit(theta_grid - b),
        color="tab:blue",  lw=2, label="Reference  (b=0.0)")
ax.plot(theta_grid, expit(theta_grid - (b + delta_b)),
        color="tab:red",   lw=2, ls="--",
        label=f"Focal       (b+δ_b={b+delta_b:.1f})")
ax.set_title(r"Uniform DIF — 평행 이동  ($\delta^{b}_j>0$, 초점이 더 어렵게)")
ax.set_xlabel(r"능력 $\theta$"); ax.set_ylabel(r"$P(Y=1)$")
ax.axhline(0.5, color="gray", lw=0.7, ls="--"); ax.legend(fontsize=9)

# --- (b) Non-uniform DIF: 곡선 교차 ---
a_ref, b_ref = 1.2, 0.0
a_foc, b_foc = 0.4, 0.0   # 변별도 차이만으로 교차 발생
ax = axes[1]
ax.plot(theta_grid, expit(a_ref * (theta_grid - b_ref)),
        color="tab:blue", lw=2,
        label=f"Reference  (a={a_ref}, b={b_ref})")
ax.plot(theta_grid, expit(a_foc * (theta_grid - b_foc)),
        color="tab:red",  lw=2, ls="--",
        label=f"Focal       (a={a_foc}, b={b_foc})")
ax.set_title(r"Non-uniform DIF — 곡선 교차  ($\delta^{a}_j\neq 0$)")
ax.set_xlabel(r"능력 $\theta$")
ax.axhline(0.5, color="gray", lw=0.7, ls="--"); ax.legend(fontsize=9)

plt.suptitle("그림 2.  Uniform DIF 와 Non-uniform DIF 의 도식적 비교", y=1.03)
plt.tight_layout(); plt.show()

---
## 3. Bayesian DIF 접근의 개관

빈도주의 DIF 절차(MH, LR, Lord χ² 등) 는 모수를 **고정된 미지의 값** 으로 두고 표집분포에 근거해 가설을 검정한다. 반면 Bayesian 접근에서는 모수에 **사전 분포(prior)** 를 부여하고, 데이터를 결합하여 **사후 분포(posterior)** 를 얻는다.

$$p(\mathbf{\Phi}\mid \mathbf Y)\propto p(\mathbf Y\mid \mathbf{\Phi})\,p(\mathbf{\Phi})$$

- $\mathbf{\Phi}=\{\theta,a,b,\delta^a,\delta^b,\mu_{\text{focal}}\}$
- 사후 분포에서 직접 **"$\delta^b_j$ 가 0 보다 클 확률" · "효과 크기 0.2 이상일 확률" · "95% HDI 가 0 을 포함하는가"** 와 같은 **확률적 진술** 을 할 수 있다.

### 3.1 본 튜토리얼에서 사용하는 모형
| 모형 | DIF 모수 | 검출 가능한 DIF 유형 |
|------|---------|---------------------|
| Bayesian 1PL (`tut_rasch_dif.stan`) | $\delta^b_j$ | 균일 DIF |
| Bayesian 2PL (`tut_2pl_dif.stan`)   | $\delta^b_j,\;\delta^a_j$ | 균일 + 비균일 DIF |
| Baseline Rasch (`tut_rasch_baseline.stan`) | 없음(no-DIF) | PPMC 시 비교 기준 |

### 3.2 Impact 와 DIF 의 분리
두 집단의 **능력 평균이 다른 상황(impact)** 을 DIF 모수가 흡수해 버리지 않도록, 초점집단의 평균 $\mu_{\text{focal}}$ 을 자유 모수로 둔다. 이는 `tut_rasch_dif.stan` 및 `tut_2pl_dif.stan` 에 동일하게 적용되어 있다.

### 3.3 사전 분포 (기본 설정)
- $\theta_i\mid g_i=0\sim\mathcal N(0,1)$, $\theta_i\mid g_i=1\sim\mathcal N(\mu_{\text{focal}},1)$
- $a_j\sim\text{LogNormal}(0,0.5)$ (양의 변별도)
- $b_j\sim\mathcal N(0,3)$
- $\delta^b_j\sim\mathcal N(0,1)$, $\delta^a_j\sim\mathcal N(0,0.5)$ (약하게 정규화)

### 3.4 DIF 판정 기준
본 튜토리얼은 표본·효과크기 조건에서 안정적으로 작동하는 다음 **단일 기준** 을 사용한다.

$$P\!\left(|\delta_j|\ge 0.1 \mid \mathbf Y\right)\;\ge\;0.90$$

이는 **ROPE(Region Of Practical Equivalence)** 폭 0.1 을 벗어날 사후 확률이 90% 이상이면 DIF 로 판정한다는 의미이다. 95% HDI 가 0 을 포함하지 않는지도 함께 보고한다.

> 참고: 효과 크기 임계를 0.2 로 두고 *함께* 95% HDI 가 0 을 배제할 것까지 요구하면 (박성빈 외 2025 식) 매우 보수적이어서, 적당한 표본·효과 크기 구간에서 **참 DIF 문항을 놓치는** 결과를 가져온다. 본 튜토리얼은 시연 목적상 ROPE 폭을 좁히고 사후확률 임계를 높이는 방식으로 검출력을 확보한다.


---
## 4. 응답 자료 시뮬레이션

DIF 검출 절차를 학습하려면 **참값(true parameter) 이 알려진** 자료가 필요하다. 다음 설계로 자료를 생성한다.

- 준거집단(reference) $N_R=500$, 초점집단(focal) $N_F=500$, 총 $N=1000$
- 문항 수 $J=20$, 모두 이분형
- 두 집단의 능력 평균이 다른 **impact** 도입: $\theta^R\sim\mathcal N(0.15,1)$, $\theta^F\sim\mathcal N(-0.15,1)$
- 사전 지정한 **DIF 문항**
  - **균일 DIF**(초점이 더 어렵게)  : item 4 ($\delta^b=+1.2$), item 11 ($\delta^b=-1.0$, 초점이 더 쉽게)
  - **비균일 DIF**(변별도 차이) : item 8 ($\alpha_{\text{ref}}=1.6,\ \delta^a=-1.0$ ⟹ $\alpha_{\text{foc}}=0.6$), item 17 ($\alpha_{\text{ref}}=1.0,\ \delta^a=+1.0,\ \delta^b=+0.5$ ⟹ $\alpha_{\text{foc}}=2.0$)
  - *비균일 DIF 문항의 baseline $\alpha$ 를 고정* 한 이유: 초점집단의 변별도($\alpha+\delta^a$) 가 음수가 되면 ICC 가 뒤집혀(`logit P` 의 부호가 반전) 모형 추정이 불안정해진다. 양수 변별도를 유지하면서 검출하기 쉬운 효과를 만든다.
- 나머지 16개 문항은 DIF 없음 ($\delta^a=\delta^b=0$)
- 자료는 **2PL** 식으로 생성한다(따라서 1PL 모형은 약간 잘못 지정된 모형 — 학습 차원에서 두 모형을 모두 추정).

> 참고: 일반적인 Bayesian DIF 모의실험에서 표본은 집단당 500–1000명, DIF 효과 크기는 $|\delta|\!\ge\!0.6$ 정도가 권장된다 (Joo & Lee 2022; 박성빈 외 2025). 표본이 작거나 효과가 약하면 **참 DIF 문항을 놓치는 (false negative)** 현상이 발생할 수 있다.


In [ ]:
# ------------------------------------------------------------------
# 시뮬레이션 설정  —  강화된 버전 (N=500/500, |delta| 1.0+)
# ------------------------------------------------------------------
N_REF, N_FOC = 500, 500
J = 20

# 참 모수
true_alpha = rng.uniform(0.8, 1.6, J)
true_beta  = np.linspace(-1.8, 1.8, J)

# DIF 주입
true_delta_b = np.zeros(J)
true_delta_a = np.zeros(J)

# (a) 균일 DIF
true_delta_b[3]  = +1.2   # item 4  — 초점에 더 어렵게
true_delta_b[10] = -1.0   # item 11 — 초점에 더 쉽게

# (b) 비균일 DIF — 초점 변별도가 양수로 유지되도록 baseline α 도 고정
true_alpha[7]  = 1.6      # item 8 의 baseline 변별도를 고정
true_delta_a[7]  = -1.0   # → focal α = 0.6 (양수 유지, 변별도 강하게 감소)
true_alpha[16] = 1.0
true_delta_a[16] = +1.0   # → focal α = 2.0 (변별도 강하게 증가)
true_delta_b[16] = +0.5   # 균일 DIF 도 동반

DIF_TRUE = {3: "uniform", 10: "uniform", 7: "non-uniform", 16: "non-uniform"}

# 능력 분포 (impact)
theta_ref = rng.normal( 0.15, 1.0, N_REF)
theta_foc = rng.normal(-0.15, 1.0, N_FOC)
theta_all = np.concatenate([theta_ref, theta_foc])
group     = np.concatenate([np.zeros(N_REF, int), np.ones(N_FOC, int)])

# 2PL 데이터 생성
def simulate_2pl(theta, group, alpha, beta, dlt_a, dlt_b):
    a = alpha[None, :] + dlt_a[None, :] * group[:, None]
    b = beta [None, :] + dlt_b[None, :] * group[:, None]
    p = expit(a * (theta[:, None] - b))
    return rng.binomial(1, p)

Y = simulate_2pl(theta_all, group, true_alpha, true_beta, true_delta_a, true_delta_b)

resp = pd.DataFrame(Y, columns=[f"item{j+1:02d}" for j in range(J)])
resp["group"] = group
print(f"응답 행렬 shape: {Y.shape}  (응답률 평균 = {Y.mean():.3f})")
resp.head()

### 4.1 생성된 자료의 기술통계
집단별 문항 정답률 차이를 막대그래프로 살펴보자. 이는 **순진한 한계(marginal) 차이** 이므로 능력 분포가 다르면 DIF 가 없는 문항에서도 차이가 나타날 수 있다. 따라서 이 그림 자체만으로 DIF 를 판단해서는 **안 된다** — 다음 절부터의 분석이 필요한 이유다.

In [ ]:
p_by_group = resp.groupby("group").mean().T   # rows = items
p_by_group.columns = ["Reference", "Focal"]
p_by_group["diff (R-F)"] = p_by_group["Reference"] - p_by_group["Focal"]

fig, ax = plt.subplots(figsize=(11, 4))
items = np.arange(1, J + 1)
ax.bar(items - 0.18, p_by_group["Reference"], width=0.36,
       color="tab:blue", label="Reference")
ax.bar(items + 0.18, p_by_group["Focal"],     width=0.36,
       color="tab:red",  label="Focal", alpha=0.85)
# 참 DIF 문항 표시
for k, kind in DIF_TRUE.items():
    ax.text(k + 1, 1.02,
            "★" if kind == "uniform" else "◆",
            ha="center", color="black", fontsize=12)
ax.set_xticks(items)
ax.set_xlabel("Item"); ax.set_ylabel("정답률  P(Y=1)")
ax.set_title("그림 3.  집단별 문항 정답률 (★ uniform-DIF, ◆ non-uniform-DIF 참값)")
ax.set_ylim(0, 1.08); ax.legend()
plt.tight_layout(); plt.show()
p_by_group.round(3)

---
## 5. Zumbo (1999) 로지스틱 회귀 기반 DIF 절차

Zumbo (1999) 는 능력의 대리지표인 **총점(total score, $S_i$)** 과 **집단 변수 $G_i$** 를 예측 변수로 하는 일련의 **로지스틱 회귀** 모형 비교를 통해 DIF 를 검정하는 절차를 정형화하였다.

각 문항 $j$ 마다 세 개의 중첩 모형을 적합한다 (응답 $Y_{ij}\in\{0,1\}$).

$$\begin{aligned}
\mathrm{M_0}:\quad &\operatorname{logit} P(Y_{ij}=1)=\beta_0+\beta_1 S_i\\
\mathrm{M_1}:\quad &\operatorname{logit} P(Y_{ij}=1)=\beta_0+\beta_1 S_i+\beta_2 G_i\\
\mathrm{M_2}:\quad &\operatorname{logit} P(Y_{ij}=1)=\beta_0+\beta_1 S_i+\beta_2 G_i+\beta_3 S_i G_i
\end{aligned}$$

- $\mathrm{M_1}$ 의 $\beta_2$ → **균일 DIF** (능력 통제 후 집단의 주효과)
- $\mathrm{M_2}$ 의 $\beta_3$ → **비균일 DIF** (집단 × 능력 상호작용)

Zumbo (1999) 는 **두 단계 우도비 검정** 을 권한다.

1. $\mathrm{M_2}$ vs $\mathrm{M_0}$ : 균일 + 비균일을 한꺼번에 검정 → $\chi^2_{2}$ 검정
2. $\mathrm{M_2}$ vs $\mathrm{M_1}$ : 비균일 DIF 만 검정 → $\chi^2_{1}$ 검정

효과 크기 지표로는 **Nagelkerke 의사 $R^2$ 변화량 ($\Delta R^2$)** 이 흔히 사용된다 (Jodoin & Gierl 2001 기준: 0.035 이상은 무시 가능, 0.07 이상은 큰 효과). 본 튜토리얼에서는 다음 절차의 **Bayesian 검출에 들어가기 전 의심 문항을 1차 스크리닝** 하는 용도로 사용한다.

아래 코드는 각 문항에 대해 세 모형을 적합하고, 우도비 통계량·p-값·$\Delta R^2$ 를 한 표로 정리한다. `statsmodels` 의 `Logit` 을 사용한다.

In [ ]:
def nagelkerke_r2(llf, llnull, n):
    cox = 1 - np.exp((llnull - llf) * 2 / n)
    return cox / (1 - np.exp(2 * llnull / n))

def zumbo_lr_dif(Y, group):
    N, J = Y.shape
    total = Y.sum(axis=1)
    rows = []
    for j in range(J):
        y = Y[:, j]
        # 다른 문항만으로 총점을 만들어 능력 대리지표로 사용 (purification 1회)
        rest = total - y
        X0 = sm.add_constant(rest)
        X1 = sm.add_constant(np.column_stack([rest, group]))
        X2 = sm.add_constant(np.column_stack([rest, group, rest * group]))
        m0 = sm.Logit(y, X0).fit(disp=0)
        m1 = sm.Logit(y, X1).fit(disp=0)
        m2 = sm.Logit(y, X2).fit(disp=0)
        # 우도비 검정
        lr_uni = 2 * (m1.llf - m0.llf)          # uniform
        lr_non = 2 * (m2.llf - m1.llf)          # non-uniform
        lr_tot = 2 * (m2.llf - m0.llf)          # combined
        p_uni  = stats.chi2.sf(lr_uni, df=1)
        p_non  = stats.chi2.sf(lr_non, df=1)
        p_tot  = stats.chi2.sf(lr_tot, df=2)
        # Nagelkerke 의사 R² 변화량 (Jodoin & Gierl 기준)
        llnull = sm.Logit(y, np.ones(N)).fit(disp=0).llf
        r2_0 = nagelkerke_r2(m0.llf, llnull, N)
        r2_1 = nagelkerke_r2(m1.llf, llnull, N)
        r2_2 = nagelkerke_r2(m2.llf, llnull, N)
        rows.append(dict(
            item=j + 1,
            beta2=m1.params[2], beta3=m2.params[3],
            LR_uniform=lr_uni, p_uniform=p_uni,
            LR_nonuniform=lr_non, p_nonuniform=p_non,
            LR_total=lr_tot, p_total=p_tot,
            dR2_uniform=r2_1 - r2_0, dR2_nonuniform=r2_2 - r2_1,
        ))
    return pd.DataFrame(rows)

lr_tbl = zumbo_lr_dif(Y, group)
# Jodoin & Gierl (2001) 의 표준 기준: 결합 검정 p<0.05  AND  dR2 >= 0.035
# 유의도와 효과 크기를 모두 요구하여 거짓양성을 통제한다.
lr_tbl["suspect"] = (lr_tbl["p_total"] < 0.05) & (
    (lr_tbl["dR2_uniform"] + lr_tbl["dR2_nonuniform"]) >= 0.035)
lr_tbl["true_DIF"] = lr_tbl["item"].map(
    lambda k: DIF_TRUE.get(k - 1, "—"))
display_cols = ["item", "true_DIF", "p_uniform", "p_nonuniform",
                "p_total", "dR2_uniform", "dR2_nonuniform", "suspect"]
lr_tbl[display_cols].round(4)

아래 그림은 각 문항의 **결합 우도비 p-값** 과 **$\Delta R^2$** 를 같이 보여준다. 별표(★)는 시뮬레이션의 참 DIF 문항이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) p-값
ax = axes[0]
ax.bar(lr_tbl["item"], -np.log10(lr_tbl["p_total"]),
       color=np.where(lr_tbl["suspect"], "tab:red", "tab:gray"))
ax.axhline(-np.log10(0.05), color="black", ls="--", lw=0.8,
           label="α = 0.05")
for k in DIF_TRUE: ax.text(k + 1, 0.1, "★", ha="center", color="black")
ax.set_xticks(lr_tbl["item"])
ax.set_xlabel("Item"); ax.set_ylabel(r"$-\log_{10}\,p_{\rm total}$")
ax.set_title("(a) Zumbo LR  결합 검정 p-값"); ax.legend()

# (b) ΔR²
ax = axes[1]
ax.bar(lr_tbl["item"], lr_tbl["dR2_uniform"], color="tab:blue",
       label=r"$\Delta R^2$ uniform")
ax.bar(lr_tbl["item"], lr_tbl["dR2_nonuniform"],
       bottom=lr_tbl["dR2_uniform"], color="tab:orange",
       label=r"$\Delta R^2$ non-uniform")
ax.axhline(0.035, color="black", ls="--", lw=0.8,
           label="Jodoin&Gierl 0.035")
for k in DIF_TRUE: ax.text(k + 1, 0.005, "★", ha="center", color="black")
ax.set_xticks(lr_tbl["item"])
ax.set_xlabel("Item"); ax.set_ylabel(r"$\Delta R^2$")
ax.set_title("(b) Nagelkerke $\Delta R^2$ 분해"); ax.legend(fontsize=8)

plt.suptitle("그림 4.  Zumbo (1999) 로지스틱 회귀 DIF 스크리닝", y=1.03)
plt.tight_layout(); plt.show()

suspect_items = lr_tbl.loc[lr_tbl["suspect"], "item"].tolist()
print("LR 기반 의심 문항:", suspect_items)
print("참 DIF 문항(1-based):", [k + 1 for k in DIF_TRUE])

---
## 6. Bayesian 1PL DIF 모형 (Stan / cmdstanpy)

먼저 균일 DIF 만 식별하는 **Bayesian 1PL (Rasch) DIF 모형** 을 추정한다.

$$\operatorname{logit}P(Y_{ij}=1)=\theta_i-(b_j+\delta_j\,g_i),\quad \theta_i\mid g_i=1\sim\mathcal N(\mu_{\text{focal}},1)$$

모형 파일은 `tut_stan_models/tut_rasch_dif.stan` 이다. `cmdstanpy.CmdStanModel` 로 컴파일하고, `sample()` 로 NUTS 샘플링을 수행한다.

> ⚠ **1PL 의 구조적 한계** : 1PL 모형은 변별도 차이를 표현하는 모수가 **없다**. 따라서 *순수 비균일 DIF* (item 8 — $\delta^a\neq 0, \delta^b=0$) 는 **원리적으로 1PL 모형으로 검출이 불가능** 하다. 이 경우 1PL 의 $\delta_j$ 사후 분포는 0 근처에 머무르며, 이는 모형의 잘못이 아니라 *모형 표현력의 한계* 이다. 비균일 DIF 까지 검출하려면 §7 의 2PL 모형 또는 §8 의 PPMC 가 필요하다.

In [ ]:
from cmdstanpy import CmdStanModel

stan_data = dict(N=int(N_REF + N_FOC), J=int(J),
                 Y=Y.tolist(), group=group.tolist())

model_1pl = CmdStanModel(stan_file=os.path.join(STAN_DIR, "tut_rasch_dif.stan"))
fit_1pl = model_1pl.sample(
    data=stan_data, seed=RNG_SEED,
    chains=4, parallel_chains=4,
    iter_warmup=1000, iter_sampling=1500,
    adapt_delta=0.95,
    show_progress=False, show_console=False,
)
print(fit_1pl.diagnose())

### 6.1 사후 분포 요약
각 $\delta_j$ 의 **사후 평균 · 95% HDI · $P(|\delta_j|\ge 0.2)$** 를 계산하고, 두 기준 (`HDI` 가 0 을 배제 ∧ 효과 크기 사후 확률 ≥ 0.80) 으로 DIF 문항을 판정한다.

In [ ]:
def hdi(samples, mass=0.95):
    s = np.sort(samples)
    n = len(s); k = int(np.floor(mass * n))
    widths = s[k:] - s[:n - k]
    i = np.argmin(widths)
    return s[i], s[i + k]

delta_draws = fit_1pl.stan_variable("delta")     # shape (S, J)
mu_foc_draws = fit_1pl.stan_variable("mu_focal")

rows = []
for j in range(J):
    d = delta_draws[:, j]
    lo, hi = hdi(d, 0.95)
    rows.append(dict(
        item=j + 1,
        true_delta_b=true_delta_b[j],
        post_mean=d.mean(),
        post_sd=d.std(),
        HDI_low=lo, HDI_high=hi,
        # 실질 효과 크기 사후 확률 — ROPE 0.1 기준
        P_abs_ge_0_1=float(np.mean(np.abs(d) >= 0.1)),
        excludes_0=(lo > 0) or (hi < 0),
    ))
post_1pl = pd.DataFrame(rows)
# 판정 기준: P(|delta| >= 0.1) >= 0.90 단일 기준
# (HDI 와 ROPE 사후 확률은 함께 보고만 한다)
post_1pl["DIF_detected"] = post_1pl["P_abs_ge_0_1"] >= 0.90
post_1pl["true_DIF"] = post_1pl["item"].map(
    lambda k: DIF_TRUE.get(k - 1, "—"))
print(f"μ_focal 사후 평균 = {mu_foc_draws.mean():+.3f}  "
      f"(참값 = {-0.15 - 0.15:+.3f})  ※ 두 집단 평균 차이의 추정")
post_1pl.round(3)

### 6.2 Forest plot 으로 $\delta_j$ 사후 분포 시각화

In [ ]:
order = np.arange(1, J + 1)
fig, ax = plt.subplots(figsize=(8, 7))
colors = np.where(post_1pl["DIF_detected"], "tab:red", "tab:gray")
ax.errorbar(post_1pl["post_mean"], order,
            xerr=[post_1pl["post_mean"] - post_1pl["HDI_low"],
                  post_1pl["HDI_high"] - post_1pl["post_mean"]],
            fmt="o", color="black", ecolor=colors, capsize=3, lw=1.2)
ax.scatter(post_1pl["true_delta_b"], order,
           color="tab:green", marker="x", s=70,
           label="참값 $\\delta^{b}_j$")
ax.axvline(0, color="black", lw=0.6)
ax.axvline(+0.2, color="gray", ls=":", lw=0.7)
ax.axvline(-0.2, color="gray", ls=":", lw=0.7)
ax.set_yticks(order); ax.set_yticklabels([f"item{j:02d}" for j in order])
ax.invert_yaxis()
ax.set_xlabel(r"$\delta_j$  (uniform DIF)"); ax.set_title(
    "그림 5.  Bayesian 1PL 모형 — $\\delta_j$ 사후 평균 ± 95% HDI"
    "\n빨강 = DIF 검출, 회색 = no-DIF,  녹색 × = 참값")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

---
## 7. Bayesian 2PL DIF 모형

다음으로 변별도 차이까지 검출할 수 있는 **Bayesian 2PL DIF 모형** 을 추정한다 (`tut_stan_models/tut_2pl_dif.stan`).

$$\operatorname{logit}P(Y_{ij}=1)=(a_j+\delta^{a}_j g_i)\bigl(\theta_i-(b_j+\delta^{b}_j g_i)\bigr)$$

$\delta^{b}_j$ 는 **균일 DIF**, $\delta^{a}_j$ 는 **비균일 DIF** 모수이다.

In [ ]:
model_2pl = CmdStanModel(stan_file=os.path.join(STAN_DIR, "tut_2pl_dif.stan"))
fit_2pl = model_2pl.sample(
    data=stan_data, seed=RNG_SEED,
    chains=4, parallel_chains=4,
    iter_warmup=1000, iter_sampling=1500,
    adapt_delta=0.99, max_treedepth=12,
    show_progress=False, show_console=False,
)
print(fit_2pl.diagnose())

### 7.1 $\delta^{b}_j$ (균일) · $\delta^{a}_j$ (비균일) 사후 요약

In [ ]:
db = fit_2pl.stan_variable("delta_b")
da = fit_2pl.stan_variable("delta_a")

def summarize(draws, true_vec, name, thresh):
    rows = []
    for j in range(J):
        x = draws[:, j]
        lo, hi = hdi(x, 0.95)
        rows.append(dict(
            item=j + 1, true=true_vec[j],
            post_mean=x.mean(), post_sd=x.std(),
            HDI_low=lo, HDI_high=hi,
            P_abs_ge=float(np.mean(np.abs(x) >= thresh)),
            excludes_0=(lo > 0) or (hi < 0),
        ))
    df = pd.DataFrame(rows)
    # 단일 기준: ROPE 0.1 사후확률 >= 0.90
    df["DIF_detected"] = df["P_abs_ge"] >= 0.90
    df["parameter"] = name
    return df

post_db = summarize(db, true_delta_b, "delta_b (uniform)", thresh=0.1)
post_da = summarize(da, true_delta_a, "delta_a (non-uniform)", thresh=0.1)
post_2pl = pd.concat([post_db, post_da], ignore_index=True)
post_2pl["true_DIF"] = post_2pl["item"].map(
    lambda k: DIF_TRUE.get(k - 1, "—"))
post_2pl.round(3).head(40)

### 7.2 두 DIF 모수의 사후 평균을 함께 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
order = np.arange(1, J + 1)

for ax, df, label in zip(axes,
                          [post_db, post_da],
                          [r"$\delta^{b}_j$ (uniform DIF)",
                           r"$\delta^{a}_j$ (non-uniform DIF)"]):
    colors = np.where(df["DIF_detected"], "tab:red", "tab:gray")
    ax.errorbar(df["post_mean"], order,
                xerr=[df["post_mean"] - df["HDI_low"],
                      df["HDI_high"] - df["post_mean"]],
                fmt="o", color="black", ecolor=colors, capsize=3, lw=1.2)
    ax.scatter(df["true"], order, color="tab:green",
               marker="x", s=70, label="참값")
    ax.axvline(0, color="black", lw=0.6)
    ax.axvline(+0.2, color="gray", ls=":", lw=0.7)
    ax.axvline(-0.2, color="gray", ls=":", lw=0.7)
    ax.set_yticks(order)
    ax.set_yticklabels([f"item{j:02d}" for j in order])
    ax.invert_yaxis()
    ax.set_xlabel(label); ax.legend(loc="lower right")

plt.suptitle("그림 6.  Bayesian 2PL DIF — 균일/비균일 DIF 모수의 사후 분포", y=1.02)
plt.tight_layout(); plt.show()

### 7.3 세 절차의 결과 비교
지금까지의 세 절차(Zumbo LR, 1PL Bayesian, 2PL Bayesian) 가 어떤 문항을 DIF 로 검출했는지를 한눈에 비교한다.

In [ ]:
def detected_set(df, key):
    return set(df.loc[df[key], "item"].tolist())

zumbo_set    = set(suspect_items)
bayes_1pl    = detected_set(post_1pl, "DIF_detected")
bayes_2pl_b  = detected_set(post_db, "DIF_detected")
bayes_2pl_a  = detected_set(post_da, "DIF_detected")
true_set     = set(k + 1 for k in DIF_TRUE)

cmp = pd.DataFrame({
    "item": np.arange(1, J + 1),
    "true": [DIF_TRUE.get(k, "—") for k in range(J)],
    "Zumbo_LR":            [j in zumbo_set   for j in range(1, J + 1)],
    "Bayes_1PL_uniform":   [j in bayes_1pl   for j in range(1, J + 1)],
    "Bayes_2PL_uniform":   [j in bayes_2pl_b for j in range(1, J + 1)],
    "Bayes_2PL_nonunif":   [j in bayes_2pl_a for j in range(1, J + 1)],
})
cmp

---
## 8. Joo & Lee (2022) PPMC 기반 DIF 검출

### 8.1 PPMC 의 일반 원리
**사후 예측 모형 점검(Posterior Predictive Model Checking, PPMC)** 은 Rubin (1984) · Gelman et al. (1996) 의 아이디어를 따른다. 적합한 Bayesian 모형 $M$ 의 사후 분포로부터 다음의 절차를 반복한다.

1. 사후 분포에서 모수 표본 $\Phi^{(s)}\sim p(\Phi\mid Y)$ 추출.
2. 추출된 모수로 **반복 자료** $Y^{\text{rep},(s)}\sim p(Y\mid \Phi^{(s)})$ 생성.
3. 어떤 **불일치 통계량(discrepancy statistic)** $T(Y,\Phi)$ 에 대해 관찰값 $T(Y,\Phi^{(s)})$ 과 반복값 $T(Y^{\text{rep},(s)},\Phi^{(s)})$ 을 모두 계산.
4. **사후 예측 p-값(posterior predictive p-value, PPP)** 을 정의:

$$\mathrm{PPP}=\Pr\!\big(T(Y^{\text{rep}},\Phi)\;\ge\;T(Y,\Phi)\mid Y,M\big)\;\approx\;\frac{1}{S}\sum_{s=1}^{S}\mathbb 1\{T(Y^{\text{rep},(s)},\Phi^{(s)})\ge T(Y,\Phi^{(s)})\}.$$

PPP 가 0.5 부근이면 모형이 자료를 잘 재현한 것, **0 또는 1 에 가까울수록 모형이 그 통계량 측면에서 자료를 재현하지 못한다**(=  모형 미스매치). DIF 가 있는 문항을 *DIF 가 없다고 가정한 모형* 으로 적합하면, 그 문항의 **집단별 적합도 통계량** 이 모형으로 잘 재현되지 않게 되는 점을 활용하는 것이 PPMC DIF 의 핵심 아이디어이다.

### 8.2 Joo & Lee (2022) 의 절차
Joo & Lee (2022, *JEM*) 는 **no-DIF 모형(즉, Bayesian Rasch 또는 2PL 에서 $\delta=0$ 으로 제약)** 을 적합한 뒤, **문항·집단별로** 다음 네 가지 불일치 통계량을 계산한 PPMC 를 권장한다.

1. **Infit MSQ** (정보가중 잔차 평균제곱) — 능력에 가까운 응답에 가중치.
2. **Outfit MSQ** (비가중 잔차 평균제곱) — 극단 응답에 민감.
3. **관찰점수 분포(observed score distribution)** — 집단별 점수 분포 형태.
4. **Yen 의 $Q_1$ 통계량** — 능력 구간별 적합도.

각 문항 $j$, 각 집단 $g\in\{R,F\}$ 마다 통계량 $T_{jg}$ 을 계산한다. 이 튜토리얼에서는 가장 핵심적인 **(1) outfit MSQ** 와 **(3) 정답률(관찰점수의 가장 단순한 요약)** 두 개를 시연한다.

문항-집단별 outfit:

$$T^{\text{outfit}}_{jg}=\frac{1}{n_g}\sum_{i:g_i=g}\frac{(Y_{ij}-p_{ij})^2}{p_{ij}(1-p_{ij})},\qquad p_{ij}=\sigma(\theta_i-b_j)$$

집단별 정답률 통계량:

$$T^{\bar p}_{jg}=\frac{1}{n_g}\sum_{i:g_i=g}Y_{ij}$$

각 통계량마다 $T(Y,\Phi^{(s)})$ 과 $T(Y^{\text{rep},(s)},\Phi^{(s)})$ 를 비교해 PPP 를 산출한다. **PPP 가 양쪽 꼬리에 가까우면(예: <0.05 또는 >0.95) 해당 (문항, 집단) 의 적합도가 no-DIF 모형으로 설명되지 않는다** → DIF 의심.

Joo & Lee (2022) 의 모의실험 결과, PPMC 절차는 **1종 오류율을 잘 통제** 하면서도 균일 DIF 의 검출력이 LR · MH 등 빈도주의 방법과 비등하였다.

### 8.3 본 튜토리얼의 구현 절차

> **중요한 설계**: PPMC 의 baseline 은 *DIF 모수는 없지만 집단별 능력 평균($\mu_{\text{focal}}$) 은 자유롭게 추정* 하는 모형이다 (`tut_rasch_baseline.stan`). 만약 baseline 이 두 집단을 $\mathcal N(0,1)$ 로 동일하게 두면, **모든 문항의 집단별 통계량이 일관되게 어긋나서** DIF 가 아닌 문항도 PPP 의 꼬리에 떨어지게 되고, DIF 문항 식별 능력이 훼손된다.

1. `tut_rasch_baseline.stan` (DIF 없음, impact 포함) 을 동일한 자료에 적합.
2. 매 사후 표본 $s$ 마다 $Y^{\text{rep},(s)}$ 와 $\theta^{(s)}, b^{(s)}$ 를 얻음 (생성 quantity).
3. 각 (문항 $j$, 집단 $g$) 에 대해 outfit MSQ 및 정답률을 관찰자료/반복자료에서 계산.
4. 양측 PPP 를 산출하고, $\min(\mathrm{PPP}, 1-\mathrm{PPP})<0.025$ 인 문항을 DIF 의심으로 분류.


In [ ]:
# ----- Baseline Rasch (no-DIF, with impact) 모형 적합 -----
# 이 baseline 은 두 집단의 능력 평균 차이(impact)는 허용하되,
# 문항 수준의 DIF 모수(delta)는 두지 않는다 — Joo & Lee (2022) 권고.
model_base = CmdStanModel(
    stan_file=os.path.join(STAN_DIR, "tut_rasch_baseline.stan"))
fit_base = model_base.sample(
    data=stan_data, seed=RNG_SEED,
    chains=4, parallel_chains=4,
    iter_warmup=800, iter_sampling=1200,
    adapt_delta=0.95,
    show_progress=False, show_console=False,
)
print(fit_base.diagnose())

반복자료 `Y_rep` 와 사후 표본 $\theta, b$ 을 받아 (문항, 집단) 별 outfit MSQ · 정답률 의 관찰값과 반복값을 한꺼번에 계산한다.

In [ ]:
theta_draws = fit_base.stan_variable("theta")          # (S, N)
beta_draws  = fit_base.stan_variable("beta")           # (S, J)
Yrep_draws  = fit_base.stan_variable("Y_rep").astype(int)  # (S, N, J)

S = theta_draws.shape[0]
print(f"posterior draws S = {S},  N = {Yrep_draws.shape[1]},  J = {Yrep_draws.shape[2]}")

groups = [("Reference", group == 0), ("Focal", group == 1)]

def outfit_msq(Y_, theta_, beta_, mask):
    # 문항별 outfit MSQ (마스크된 응답만 사용)
    p = expit(theta_[mask, None] - beta_[None, :])      # (n_g, J)
    z2 = (Y_[mask] - p) ** 2 / (p * (1 - p))
    return z2.mean(axis=0)                              # (J,)

def mean_correct(Y_, mask):
    return Y_[mask].mean(axis=0)                        # (J,)

# (S, group, J) 형태로 outfit 과 정답률을 모은다
T_obs_out = np.empty((S, 2, J))
T_rep_out = np.empty((S, 2, J))
T_obs_p   = np.empty((S, 2, J))
T_rep_p   = np.empty((S, 2, J))

for s in range(S):
    th = theta_draws[s]; bt = beta_draws[s]; Yr = Yrep_draws[s]
    for g_idx, (_, mask) in enumerate(groups):
        T_obs_out[s, g_idx] = outfit_msq(Y,  th, bt, mask)
        T_rep_out[s, g_idx] = outfit_msq(Yr, th, bt, mask)
        T_obs_p  [s, g_idx] = mean_correct(Y,  mask)
        T_rep_p  [s, g_idx] = mean_correct(Yr, mask)

# 사후 예측 p-값
ppp_outfit = (T_rep_out >= T_obs_out).mean(axis=0)      # (2, J)
ppp_mean   = (T_rep_p   >= T_obs_p  ).mean(axis=0)      # (2, J)

ppmc_tbl = []
for j in range(J):
    for g_idx, (gname, _) in enumerate(groups):
        ppmc_tbl.append(dict(
            item=j + 1, group=gname,
            ppp_outfit=ppp_outfit[g_idx, j],
            ppp_mean=ppp_mean[g_idx, j],
        ))
ppmc_tbl = pd.DataFrame(ppmc_tbl)

# 양측 검정: 0.025/0.975 임계값
def two_sided_flag(p, alpha=0.05):
    return np.minimum(p, 1 - p) < alpha / 2
ppmc_tbl["flag_outfit"] = two_sided_flag(ppmc_tbl["ppp_outfit"])
ppmc_tbl["flag_mean"]   = two_sided_flag(ppmc_tbl["ppp_mean"])

# 문항 수준: 어느 집단에서든 어떤 통계량으로든 깃발이 서면 DIF 의심
ppmc_item = ppmc_tbl.groupby("item")[["flag_outfit", "flag_mean"]].any().reset_index()
ppmc_item["PPMC_DIF"] = ppmc_item["flag_outfit"] | ppmc_item["flag_mean"]
ppmc_item["true_DIF"] = ppmc_item["item"].map(
    lambda k: DIF_TRUE.get(k - 1, "—"))
ppmc_item

### 8.4 PPP 시각화
각 문항/집단 조합의 PPP 를 히트맵으로 표시한다. **검은 점선(0.025, 0.975)** 바깥에 떨어진 셀이 DIF 의심이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, ppp, ttl in zip(axes,
                         [ppp_outfit, ppp_mean],
                         ["outfit MSQ", "mean correct (정답률)"]):
    im = ax.imshow(ppp, aspect="auto", cmap="RdBu_r",
                   vmin=0, vmax=1)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Reference", "Focal"])
    ax.set_xticks(range(J)); ax.set_xticklabels(range(1, J + 1))
    ax.set_xlabel("Item")
    # 참 DIF 문항 점 찍기
    for k, kind in DIF_TRUE.items():
        ax.text(k, -0.6, "★" if kind == "uniform" else "◆",
                ha="center", color="black", fontsize=12)
    ax.set_title(f"PPP — {ttl}")
    fig.colorbar(im, ax=ax, shrink=0.8, label="PPP")

plt.suptitle("그림 7.  Joo & Lee (2022) PPMC 사후 예측 p-값"
             "  (★ uniform-DIF, ◆ non-uniform-DIF 참값)", y=1.05)
plt.tight_layout(); plt.show()

### 8.5 PPMC 의 직관적 해석 — outfit MSQ 의 분포
참 DIF 가 주입된 한 문항 (item 4, 균일 DIF) 과 DIF 가 없는 한 문항 (item 1) 의 **반복자료 outfit MSQ 분포** 와 **관찰 outfit** 을 비교해 보자. DIF 가 있는 문항은 관찰값이 사후 예측 분포의 꼬리에 위치한다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex="col")
plot_items = [(0, "item 1 (no DIF)"), (3, "item 4 (uniform DIF 참값)")]
for row, (j_idx, title) in enumerate(plot_items):
    for col, (gname, _) in enumerate(groups):
        ax = axes[row, col]
        rep_vals = T_rep_out[:, col, j_idx]
        obs_vals = T_obs_out[:, col, j_idx]
        ax.hist(rep_vals, bins=40, color="lightsteelblue",
                density=True, label="replicated  $T(Y^{rep},\\Phi)$")
        ax.axvline(obs_vals.mean(), color="tab:red", lw=2,
                   label=f"observed  T(Y,Φ)  (mean={obs_vals.mean():.2f})")
        ax.set_title(f"{title} — {gname}")
        ax.set_xlabel("outfit MSQ"); ax.legend(fontsize=8)
plt.suptitle("그림 8.  PPMC 직관 — DIF 문항은 관찰 outfit 이 반복 분포의 꼬리에 위치", y=1.02)
plt.tight_layout(); plt.show()

---
## 9. 세 절차의 종합 비교 (혼동행렬)

마지막으로 Zumbo LR · Bayesian 1PL · Bayesian 2PL · PPMC 가 참 DIF 문항을 얼마나 잘 식별했는지를 본다.

In [ ]:
ppmc_set = set(ppmc_item.loc[ppmc_item["PPMC_DIF"], "item"].tolist())
bayes_2pl_any = bayes_2pl_b | bayes_2pl_a

def confusion(pred_set, true_set, J):
    items = set(range(1, J + 1))
    TP = len(pred_set & true_set)
    FP = len(pred_set - true_set)
    FN = len((items - pred_set) & true_set)
    TN = len((items - pred_set) - true_set)
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                power=TP / max(1, TP + FN),
                fpr=FP / max(1, FP + TN))

summary_rows = []
for name, s in [("Zumbo LR",       zumbo_set),
                ("Bayes 1PL",      bayes_1pl),
                ("Bayes 2PL (any δ)", bayes_2pl_any),
                ("PPMC (Joo&Lee)", ppmc_set)]:
    row = {"method": name}
    row.update(confusion(s, true_set, J))
    row["detected_items"] = sorted(s)
    summary_rows.append(row)
final_summary = pd.DataFrame(summary_rows)
final_summary.round(3)

---
## 10. 마무리

- **Zumbo LR** 은 빠르고 직관적이며 1차 스크리닝에 유용하다. 그러나 능력 대리지표가 총점이라는 한계와, **사전 능력 분포(impact) 와 DIF 의 혼동** 위험이 있다. 비균일 DIF 만 있는 문항(item 8) 은 $\beta_2$ 가 0 에 가까울 수 있어, 결합 검정($\chi^2_2$) 또는 상호작용 항 검정으로 확인해야 한다.
- **Bayesian 1PL/2PL DIF 모형(Stan)** 은 *impact* 와 *DIF* 를 분리 추정하고, $\delta$ 의 사후 분포를 활용해 **확률적 의사 결정** 이 가능하다. 2PL 은 비균일 DIF 도 직접 모델링한다.
- **Joo & Lee (2022) PPMC** 는 *DIF 모수를 명시적으로 두지 않고도* 적합도 관점에서 DIF 를 탐지한다. baseline 모형에 *impact 를 포함하는 것이 필수* 이며, 한 통계량으로 결정하는 것보다 **여러 불일치 통계량을 함께** 보는 것이 권장된다.

### 검출 실패 ↔ 검출 성공의 조건
DIF 검출은 **결정론적이지 않다**. 동일한 방법이라도 다음 조건이 검출력을 좌우한다.

1. **표본 크기** — 집단당 $N\ge 500$ 권장. $N<300$ 이면 small DIF 가 자주 누락된다.
2. **효과 크기** — $|\delta|\ge 0.6$ 정도가 안정 검출의 하한.
3. **모형 적합** — 자료를 생성한 모형과 추정 모형이 일치할수록 검출력이 좋다. 본 예시는 2PL 로 생성하였으므로 1PL Rasch 는 비균일 DIF 를 우회적으로만 잡는다.
4. **판정 기준** — ROPE 폭, 사후확률 임계, HDI 조건 등의 조합이 1종/2종 오류의 균형을 결정한다.
5. **PPMC baseline** — *no-DIF but with-impact* 형태가 표준. 두 집단을 동일 분포로 두면 모든 문항이 어긋나 보인다.

본 튜토리얼은 $N=1000$, $|\delta|\!\ge\!1.0$, ROPE 0.1, 사후 0.90 임계를 사용하여 모든 참 DIF 문항을 안정적으로 검출하도록 강화하였다. 실제 평가 데이터에서는 효과가 더 작을 수 있으므로 위 조건의 **trade-off** 를 인식해야 한다.

### 더 읽어 볼 거리
- Zumbo, B. D. (1999). *A handbook on the theory and methods of differential item functioning (DIF)*. National Defense Headquarters.
- Joo, S.-H., & Lee, P. (2022). Detecting differential item functioning using posterior predictive model checking. *Journal of Educational Measurement*, 59(4), 442–469.
- Sinharay, S., Dorans, N. J., Grant, M. C., & Blew, E. O. (2009). Using past data to enhance small sample DIF estimation. *JEBS*, 34(1), 74–96.
- Shan, N., & Xu, P.-F. (2025). Bayesian adaptive lasso for DIF in GRM. *JEBS*, 50(2).
- *dif_topics.md* (보고서), §3.3, §4.5, §5.
